# Financial RAG — CUAD clause-retrieval fine-tune

Runs the full Option A pipeline on a GPU (Colab T4 / Kaggle): build leakage-free
CUAD pairs, fine-tune `bge-base` and a cross-encoder reranker, then measure base vs
fine-tuned vs fine-tuned+reranker on per-contract clause retrieval.

**Runtime → GPU** before running. ~15–25 min end to end.

In [ ]:
# transformers==4.46.3 is the newest release still exporting find_pruneable_heads_
# and_indices, which sentence-transformers' fit() imports; its tokenizers pin
# (0.20.x) ships a prebuilt wheel, so nothing compiles from source here.
!pip -q install -U "sentence-transformers>=3.3,<4" "transformers==4.46.3" "accelerate>=0.26"

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"  # sentence-transformers' fit() auto-logs to
                                        # W&B otherwise and blocks on an interactive
                                        # prompt with no account configured.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # pin to one GPU: on a 2xT4 session,
                                           # DataParallel gathers outputs onto GPU 0
                                           # and can OOM it even though each GPU
                                           # alone has room for this workload.

import random, re, io, json, zipfile, urllib.request, numpy as np, torch
from collections import defaultdict
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.cross_encoder import CrossEncoder
from torch.utils.data import DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
random.seed(0)
N_TRAIN, N_TEST, N_NEG = 120, 20, 3

## 1. Build leakage-free pairs and eval set (disjoint train/test contracts)

In [ ]:
# CUAD_v1.json is SQuAD-format: data -> {title, paragraphs -> {context, qas}}.
print("downloading CUAD (~106 MB) ...")
buf = urllib.request.urlopen("https://zenodo.org/records/4595826/files/CUAD_v1.zip?download=1").read()
raw = json.loads(zipfile.ZipFile(io.BytesIO(buf)).read("CUAD_v1/CUAD_v1.json"))
by_title, ctx = defaultdict(list), {}
for art in raw["data"]:
    for para in art["paragraphs"]:
        ctx[art["title"]] = para["context"]
        for qa in para["qas"]:
            if qa.get("answers"):
                by_title[art["title"]].append((qa["question"], qa["answers"][0]["text"]))
print("contracts:", len(by_title), "| answered pairs:", sum(len(v) for v in by_title.values()))

titles = [t for t in by_title if len(by_title[t]) >= 5]
random.shuffle(titles)
train_titles, test_titles = titles[:N_TRAIN], titles[N_TRAIN:N_TRAIN+N_TEST]

train_pairs = []  # (query, positive, [negatives]) — negatives are other-clause answers, same contract
for t in train_titles:
    answers = [a for _, a in by_title[t]]
    for q, a in by_title[t]:
        negs = [x for x in answers if x != a]; random.shuffle(negs)
        train_pairs.append((q, a, negs[:N_NEG]))

test_golden = [(q, a, t) for t in test_titles for q, a in by_title[t]]

def chunks_of(text, size=1500, overlap=200):
    out, i = [], 0
    while i < len(text):
        out.append(text[i:i+size]); i += size - overlap
    return out
test_chunks = [(t, c) for t in test_titles for c in chunks_of(ctx[t])]

print(f"train_pairs={len(train_pairs)}  test_questions={len(test_golden)}  test_chunks={len(test_chunks)}")

## 2. Fine-tune the embedding model (bge-base)

In [ ]:
examples = [InputExample(texts=[q, p, n[0]] if n else [q, p]) for q, p, n in train_pairs]
emb = SentenceTransformer("BAAI/bge-base-en-v1.5", device=DEVICE)
loader = DataLoader(examples, shuffle=True, batch_size=8)  # CUAD spans are long;
                                                            # a single T4 GPU is
                                                            # the safe target here
emb.fit(train_objectives=[(loader, losses.MultipleNegativesRankingLoss(emb))],
        epochs=2, warmup_steps=int(0.1*len(loader)*2), show_progress_bar=True)
emb.save("bge-cuad-ft")

## 3. Train the cross-encoder reranker

In [ ]:
ce_examples = []
for q, p, negs in train_pairs:
    ce_examples.append(InputExample(texts=[q, p], label=1.0))
    for n in negs:
        ce_examples.append(InputExample(texts=[q, n], label=0.0))
ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", num_labels=1, device=DEVICE)
ce_batch = 16
ce.fit(DataLoader(ce_examples, shuffle=True, batch_size=ce_batch),
       epochs=1, warmup_steps=int(0.1*len(ce_examples)/ce_batch), show_progress_bar=True)
ce.save("reranker-cuad")

## 4. Evaluate (per-contract clause retrieval)

In [ ]:
def norm(t): return re.sub(r"\s+", "", t.lower())
def relevant(chunk, ans): return norm(ans[:150]) in norm(chunk)

def _sync():
    if torch.cuda.is_available(): torch.cuda.synchronize()

def metrics(model, reranker=None, k_list=(1,5,10)):
    # encode test chunks once, grouped per contract
    per_title = defaultdict(list)
    for t, c in test_chunks: per_title[t].append(c)
    enc = {t: model.encode(cs, convert_to_numpy=True, normalize_embeddings=True) for t, cs in per_title.items()}

    # warm-up: exclude one-off CUDA kernel compilation / cache fill from the timed pass
    q0, _, t0 = test_golden[0]
    model.encode(q0, convert_to_numpy=True, normalize_embeddings=True)
    if reranker is not None:
        reranker.predict([[q0, per_title[t0][0]]])
    _sync()

    agg = {k: [0.0,0.0,0.0] for k in k_list}
    embed_times, rerank_times = [], []
    for q, a, t in test_golden:
        cs = per_title[t]
        _sync(); t_start = time.perf_counter()
        qv = model.encode(q, convert_to_numpy=True, normalize_embeddings=True)
        _sync(); embed_times.append(time.perf_counter() - t_start)

        order = np.argsort(-(enc[t] @ qv))[:10]
        ranked = [cs[i] for i in order]
        if reranker is not None:
            _sync(); t_start = time.perf_counter()
            scores = reranker.predict([[q, c] for c in ranked])
            _sync(); rerank_times.append(time.perf_counter() - t_start)
            ranked = [c for _, c in sorted(zip(scores, ranked), key=lambda x: -x[0])]

        gold = [i for i, c in enumerate(ranked) if relevant(c, a)]
        for k in k_list:
            hits = [i for i in gold if i < k]
            agg[k][0] += 1.0 if hits else 0.0                      # recall@k (>=1 gold in top-k)
            agg[k][1] += (1.0/(hits[0]+1)) if hits else 0.0        # MRR@k
            dcg = sum(1.0/np.log2(i+2) for i in hits)
            idcg = sum(1.0/np.log2(i+2) for i in range(min(len(gold),k)))
            agg[k][2] += (dcg/idcg) if idcg else 0.0               # nDCG@k
    n = len(test_golden)
    result = {k: [v/n for v in vals] for k, vals in agg.items()}
    latency = {"embed_ms": np.array(embed_times) * 1000}
    if rerank_times:
        latency["rerank_ms"] = np.array(rerank_times) * 1000
    return result, latency

import time
base = SentenceTransformer("BAAI/bge-base-en-v1.5", device=DEVICE)
ft = SentenceTransformer("bge-cuad-ft", device=DEVICE)
rows = [("base bge-base", *metrics(base)),
        ("fine-tuned bge-base", *metrics(ft)),
        ("fine-tuned + trained reranker", *metrics(ft, ce))]

print(f"{'config':32} {'r@1':>6} {'r@5':>6} {'r@10':>6} {'nDCG@10':>8}")
for label, m, _ in rows:
    print(f"{label:32} {m[1][0]:6.3f} {m[5][0]:6.3f} {m[10][0]:6.3f} {m[10][2]:8.3f}")

print()
print(f"{'config':32} {'embed p50':>10} {'embed p95':>10} {'rerank p50':>11} {'rerank p95':>11} {'total p50':>10}")
for label, _, lat in rows:
    e = lat['embed_ms']; ep50, ep95 = np.percentile(e, [50, 95])
    if 'rerank_ms' in lat:
        r = lat['rerank_ms']; rp50, rp95 = np.percentile(r, [50, 95])
        print(f"{label:32} {ep50:9.1f}ms {ep95:9.1f}ms {rp50:10.1f}ms {rp95:10.1f}ms {ep50+rp50:9.1f}ms")
    else:
        print(f"{label:32} {ep50:9.1f}ms {ep95:9.1f}ms {'—':>11} {'—':>11} {ep50:9.1f}ms")